In [55]:
import numpy as np
import pandas as pd
import time
from typing import Tuple, List, Dict
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pathlib import Path #to keep importing datasets from relative paths consistent across OS

### Built Matrix Factorization Recommender 

In [ ]:
class MF_Recommender:
    '''
    Matrix Factorization Recommender System using SGD (Stochastic Gradient Descent) 

    Model decomposes user-movie rating matrix into 2 separate lower ranked and denser matrices
    -P (User latent factors): shape (m users, k factors)
    -Q (Movie latent factors): shape (n users, k factors) 

    Predicted ratings is R_hat = P@Q.T + biases

    '''

    def __init__(self, n_factors: int, learning_rate: float, regularization: float, epochs: int, 
                 batch_size = 1024, Output_Status: bool = True, use_bias: bool = True, 
                 use_latent_features: bool = True, use_content_features: bool = False):
                 
        '''
        n_factors = int # of latent factors
        learning_rate = float for the stochastic gradient descent optimization
        regularization = float (L2 regularization param to prevent overfitting)
        epochs = int # of passes through training data
        Output_Status = whether to print training progress (ALWAYS SET TO TRUE)
        use_bias = bool whether to use bias terms in the model 
        use_latent_features = bool turn model into baseline recommender
        use_content_features = bool whether to use content features (rating count, tags, etc)
        '''
    
        #initialization vals
        self.n_factors = n_factors #k latent factors
        self.learning_rate = learning_rate #.01 float
        self.regularization = regularization #.01 float
        self.epochs = epochs  #int
        self.Output_Status = True #ALWAYS SET TO TRUE
        self.use_bias = use_bias #T/F
        self.use_latent_features = use_latent_features #T/F
        self.use_content_features = use_content_features #T/F
        self.batch_size = batch_size
    
        '''
        user_factors = P matrix: user latent factors
        movie_factors = Q matrix: movie latent factors
        user_biases = user bias term
        movie_biases = movie bias term
        global_mean = global mean rating
        user_content_weights = use in Hybrid model 
        movie_content_weights = use in Hybrid model
    
        '''
        #model params
    
        self.user_factors = None #P matrix: user latent factors
        self.movie_factors = None #Q matrix: movie latent factors
        self.user_biases = None #user bias terms
        self.movie_biases = None #movie bias terms
        self.global_mean = None #global mean rating
    
        #For hybrid model
        self.user_content_weights = None 
        self.movie_content_weights = None
        
        #Map for movie/user for matrix
        self.user_id_to_idx = {}
        self.movie_id_to_idx = {}
        self.idx_to_user_id = {}
        self.idx_to_movie_id = {}
        
        #training history
        self.train_losses = []
        self.train_mae_losses = []
        self.val_losses = []
        self.val_mae_losses = []

        #evaluation
        self.precision_at_k = None
        self.recall_at_k = None
        
    def initialize_model_params(self, users: int, movies: int, user_features: int = 0, movie_features: int = 0):
        '''
        Initialize the parameters for the model with near 0 values to make 
        sure model doesn't have dead gradients
    
        users: number of users
        movies: number of movies
        user_features: number of user features (may not have any)
        movie_features: number of movie features (may not have any)
        '''
        if self.use_latent_features == True:
            self.user_factors = np.random.normal(0, .001, (users, self.n_factors)) #mean 0, std of .001
            self.movie_factors = np.random.normal(0, .001, (movies, self.n_factors)) #mean 0, std of .001

        else: ############### remove logic if it breaks #######################
            self.user_factors = np.random.normal(0, 0.0, (users, self.n_factors))
            self.movie_factors = np.random.normal(0, 0.0, (movies, self.n_factors))
            
        if self.use_bias == True:
            #print("I'm in use bias")
            #initialize bias to 0 for users and movies
            self.user_biases = np.zeros(users)
            self.movie_biases = np.zeros(movies)
    
        if self.use_content_features == True: 
            #initialize content feature weights
            #in hybrid model will interact with latent features
            if user_features > 0: 
                self.user_content_weights = np.random.normal(0, .001, (user_features, self.n_factors))
            if movie_features > 0: 
                self.movie_content_weights = np.random.normal(0, .001, (movie_features, self.n_factors))
                
    def initialize_mappings(self, user_ids: np.ndarray, movie_ids: np.ndarray):
        '''
        The engine that makes the matrix factorization memory and computationally efficient
    
        user_ids: numpy array of unique user_ids
        movie_ids: numpy array of unique movie_ids
        
        '''
        #make the mapping for users go forward and backwards for faster search
        for idx, user_id in enumerate(np.unique(user_ids)):
            self.user_id_to_idx[user_id] = idx
            self.idx_to_user_id[idx] = user_id
        
        #make the mapping for movies go forward and backwards for faster search
        for idx, movie_id in enumerate(np.unique(movie_ids)):
            self.movie_id_to_idx[movie_id] = idx
            self.idx_to_movie_id[idx] = movie_id 

    def get_content_features(self, ratings_df: pd.DataFrame, tags_df):
        '''
        To be completed later
        '''


    def predict_single_rating(self, user_idx: int, movie_idx: int, user_features= None, movie_features= None) -> float:
        '''
        SGD and any predictions cannot be trained in batches so each individual 
        user movie combo must be trained sequentially. Testing can be predicted 
        in batches and vectorized, though. 
    
        Predict single rating for user-movie pair
    
        Vanilla MF:
        rating = global mean + user bias + movie bias + user factors @ movie factors.T
    
        Hybrid model: 
        rating += (user features @ user content weights) @ (movie features @ movie content weights).T
    
        user_idx: int. user index in matrix
        movie_idx: int. movie index in matrix
    
        for Hybrid Model
        user_features: numpy array. user content features
    
        movie features: numpy array. movie content features
    
        returns float prediction for user movie combo
        '''
    
        prediction = self.global_mean
    
        #if self.user_biases is None and movie_biases is None:
        #    user_biases = np.zeros(users)
        #    movie_biases = np.zeros(movies)
        #add bias if used
        if self.use_bias == True: 
            prediction += self.user_biases[user_idx] + self.movie_biases[movie_idx]

        if self.use_latent_features == True: 
            #add latent factors
            prediction += self.user_factors[user_idx]@self.movie_factors[movie_idx]
    
        #add content features for hybrid model if used
        if self.use_content_features == True and user_features is not None and movie_features is not None: 
            #to be filled later
            1
            
        return prediction

    def SGD(self, user_idx: int, movie_idx: int, rating: float, user_features= None, movie_features= None):
        '''
        perform a single SGD update for one user-movie rating
    
        user_idx: int. User index in the matrix
        movie_idx: int. Movie index in the matrix
        rating: float. Actual movie rating
    
        Hybrid Model
        user_features: numpy array. User Content features
        movie_features: numpy array. Movie Content features
        '''
    
        #calculate prediciton error
        prediction = self.predict_single_rating(user_idx, movie_idx, user_features, movie_features)
        prediction_error = rating - prediction
    
        #store old values to update all simultaneously
        old_user_factors = self.user_factors[user_idx].copy()
        old_movie_factors = self.movie_factors[movie_idx].copy()
    
        #update latent factors using gradient descent
        self.user_factors[user_idx] += self.learning_rate * (prediction_error * old_movie_factors - self.regularization * old_user_factors)
    
        self.movie_factors[movie_idx] += self.learning_rate * (prediction_error * old_user_factors - self.regularization * old_movie_factors)
    
        #update biases if used
        if self.use_bias == True: 
            ############################################ Check ##################
            #may turn off if the bias update doesn't need regularization
            ####################################################################
            self.user_biases[user_idx] += self.learning_rate * prediction_error
            self.movie_biases[movie_idx] += self.learning_rate * prediction_error
    
        #update content feature weights for Hybrid model
        if self.use_content_features == True and user_features is not None and movie_features is not None: 
            1

    def fit_model(self, ratings_df: pd.DataFrame, tags_df: None, val_split: float=.2):
        '''
        Train the MF model using SGD
    
        ratings_df: the ratings dataframe
        tags_df: the tags for the hybrid model
        val_split: float. the percent split for data validation
        
        '''
        
        #create train/validation split
        train_df, val_df = train_test_split(ratings_df, test_size=val_split, random_state = 158)
    
        #create user/movie mappings
        self.initialize_mappings(ratings_df['userId'].values, ratings_df['movieId'].values)
    
        #get global mean
        self.global_mean = train_df['rating'].mean() 
    
        #get content features for hybrid model
        user_features = None
        movie_features = None
        if self.use_content_features == True:
            1
        else:
            user_features = 0
            movie_features = 0
        
        #initialize parameters of model
        total_users = len(self.user_id_to_idx)
        total_movies = len(self.movie_id_to_idx)
        self.initialize_model_params(total_users, total_movies, user_features, movie_features)

        ###################### MINI BATCH SGD ###################
        
        user_indices = np.array([self.user_id_to_idx[uid] for uid in train_df['userId'].values])
        movie_indices = np.array([self.movie_id_to_idx[mid] for mid in train_df['movieId'].values])
        ratings = train_df['rating'].values
        
        len_ratings = len(ratings)
        
        n_batches = (len_ratings + self.batch_size - 1)// self.batch_size #make mini batches
    
        #start training
        if self.Output_Status == True: 
            print(f'Training started with Mini Batch SGD (batch size={self.batch_size})')
            print(f'Data has {total_users} Total Users, {total_movies} Total movies, and {self.n_factors} K dimensions')
            print(f'Total Epochs is {self.epochs}, Learning rate is {self.learning_rate}, and {n_batches} batches per epoch')
            print('')

        epoch_mae_loss = []
        epoch_loss = [] 
        
        start = time.time()
        
        for epoch in range(self.epochs):
            #randomize the data for sgd
            #shuffled_training_data = train_df.sample(frac=1, random_state=epoch)
            if (epoch % 10 == 0 or epoch == self.epochs -1 ):
                print(f'Starting fit for epoch {epoch+1}/{self.epochs}')
            
            indices = np.random.permutation(len_ratings)
            user_indices_shuffled = user_indices[indices]
            movie_indices_shuffled = movie_indices[indices]
            ratings_shuffled = ratings[indices]

            total_mae_loss = 0
            total_loss = 0 

            #### implement Mini batch SGD 
            for batch_begin in range(0, len_ratings, self.batch_size):
                batch_end = min(batch_begin + self.batch_size, len_ratings)

                batch_users = user_indices_shuffled[batch_begin:batch_end]
                batch_movies = movie_indices_shuffled[batch_begin:batch_end]
                batch_ratings = ratings_shuffled[batch_begin:batch_end]

                #vectorize the predictions for batch
                prediction = (self.global_mean + self.user_biases[batch_users] + self.movie_biases[batch_movies])

                #dot product
                for i in range(len(batch_users)):
                    prediction[i] += self.user_factors[batch_users[i]]@self.movie_factors[batch_movies[i]]

                #calculate error for batches
                batch_error = batch_ratings - prediction

                #calc mae loss
                total_mae_loss += np.sum(np.absolute(batch_error))
                
                total_loss += np.sum(batch_error ** 2)

                #vectorize the updates
                for i in range(len(batch_users)):
                    u_idx = batch_users[i]
                    m_idx = batch_movies[i]
                    error = batch_error[i]

                    #update with momentum
                    u_grad = error * self.movie_factors[m_idx] - self.regularization * self.user_factors[u_idx]
                    m_grad = error * self.user_factors[u_idx] - self.regularization * self.movie_factors[m_idx]

                    self.user_factors[u_idx] += self.learning_rate * u_grad
                    self.movie_factors[m_idx] += self.learning_rate * m_grad
                    self.user_biases[u_idx] += self.learning_rate * error 
                    self.movie_biases[m_idx] += self.learning_rate * error

            #calculate loss
            mae = ((1.0 * total_mae_loss)/ len_ratings)
            rmse = np.sqrt(total_loss / len_ratings)

            epoch_mae_loss.append(mae)
            epoch_loss.append(rmse)
            if (epoch % 10 == 0 or epoch == self.epochs -1 ):
                print(f'Finished fitting Epoch {epoch+1}')
            
            
        #Calc epoch loss metrics
        epoch_mae_train_loss = np.mean(epoch_mae_loss)
        epoch_train_loss = np.mean(epoch_loss)
        self.train_mae_losses.append(epoch_mae_train_loss)
        self.train_losses.append(epoch_train_loss)


        end = time.time()

        print(f'Finished all Epochs. Total time elapsed {(end - start)/60 : .2f} minutes')
        
        #validation set loss calculations
        val_predictions = []
        val_actual = []
        for _, row in val_df.iterrows():
            if row['userId'] in self.user_id_to_idx and row['movieId'] in self.movie_id_to_idx:
                user_idx = self.user_id_to_idx[row['userId']]
                movie_idx = self.movie_id_to_idx[row['movieId']]
                v_prediction = self.predict_single_rating(user_idx, movie_idx, user_features, movie_features)
                val_predictions.append(v_prediction)
                val_actual.append(row['rating'])
    
        val_rmse_loss = mean_squared_error(val_actual, val_predictions)
        val_mae_loss = mean_absolute_error(val_actual, val_predictions)
        
        self.val_losses.append(val_rmse_loss)
        self.val_mae_losses.append(val_mae_loss)
    
        #update learning rate automatically (may be useful)
        #if epoch %10 == 0 and epoch > 0: 
        #    learning_rate *= .9

        end2 = time.time()
        print(f'Finished fitting. Total time elapsed {(end2 - start)/60 : .2f} minutes')
        
        #show progress on epochs
        if self.Output_Status == True and (epoch % 10 == 0 or epoch == self.epochs -1 ):
            print(f'Epoch {epoch + 1}/{self.epochs}: Training RMSE loss is {epoch_train_loss: .6f} and Validation RMSE loss is {val_rmse_loss: .6f}')
            print(f'Training MAE loss is {epoch_mae_train_loss: .6f} and Validation MAE loss is {val_mae_loss: .6f}')
            
    def predict_for_batch(self, user_ids:  np.ndarray, movie_ids: np.ndarray) -> np.ndarray:
        '''
        Prediction for batchs of user-movies combos. Includes code to remove cold start problem results
    
        user_ids: numpy array of user Ids
        movie_ids: numpy array of movie Ids
    
        return: numpy array of predicted ratings
        '''

        ############################### 

        '''
        ### Remove invalid movie ratings #####
        #  create a set of missing movies not in training
        missing_movies = set(df['movieId']) - set(train_df['movieId'])
        if missing_movies: # if we have missing movies
            # Add one random row per missing movie
            movies_to_add = (df[df['movieId'].isin(missing_movies)] # create the filtered df
                .groupby('movieId', as_index=False, sort=False) # group by movie id
                .apply(lambda x: x.sample(1, random_state=random_state)) # grab one sample from each group
                .reset_index(drop=True))
            # concat with the training set, we might have duplicates on the test set
        train_df = pd.concat([train_df, movies_to_add], ignore_index=True)
        '''
        #if user/movie in test did not appear in train, have handler
        valid_boolean_mask = np.array([uid in self.user_id_to_idx and mid in self.movie_id_to_idx
                                       for uid, mid in zip(user_ids, movie_ids)])

        predictions = np.full(len(user_ids), self.global_mean)

        #if there are no valid user/movies
        if not np.any(valid_boolean_mask):
            return predictions

        #get valid user/movie Ids
        valid_user_ids = user_ids[valid_boolean_mask]
        valid_movie_ids = movie_ids[valid_boolean_mask]

        #convert to indices 
        user_indices = np.array([self.user_id_to_idx[uid] for uid in valid_user_ids])
        movie_indices = np.array([self.movie_id_to_idx[mid] for mid in valid_movie_ids])

        #vectorized prediction computation 
        valid_predictions = np.full(len(user_indices), self.global_mean)
        
        if self.use_bias ==True: 
            #vectorized biases are added
            valid_predictions += self.user_biases[user_indices]
            valid_predictions += self.movie_biases[movie_indices]

        if self.use_latent_features == True:
        #apparently einsum is especially well built for computing batch dot products with shifting dimensions
            latent_predictions = np.einsum('ij, ij->i', 
                                        self.user_factors[user_indices], 
                                        self.movie_factors[movie_indices])

            valid_predictions += latent_predictions

        #user boolean mask to output only valid predictions
        predictions[valid_boolean_mask] = valid_predictions

        
        return predictions


        

    def recommend_movies(self, user_id: int, n_recommendations: int=3, 
                         exclude_already_watched: bool = True, already_watched = None) -> List[Tuple[int, float]]:
        '''
        Make the top N recommendations for any given user
        '''
        #edge case in case its a new user
        #if user_id not in user_id_to_idx: 
        #    return []

        if user_id not in self.user_id_to_idx:
            print("Not in user_id_to_idx")
            return []
        
        user_idx = self.user_id_to_idx[user_id]
    
        #generate predictions for all movies
        movie_predictions = []
        for movie_id, movie_idx in self.movie_id_to_idx.items():
            #skip if excluding movies already watched
            if exclude_already_watched == True and already_watched and movie_id in already_watched: 
                continue
    
            pred = self.predict_single_rating(user_idx, movie_idx)
            movie_predictions.append((movie_id, pred))
    
        #sort by predicted rating (descending order) and return top N 
        movie_predictions.sort(key = lambda x: x[1], reverse = True)
        return movie_predictions[:n_recommendations]

    def simple_evaluation(self, test_set):
        user_ids, movie_ids = test_set['userId'], test_set['movieId']
        #make predictions
        v_mf_predictions = self.predict_for_batch(user_ids, movie_ids)
        
        #limit range of predictions to only be between .5-5.0
        v_mf_predictions = np.clip(v_mf_predictions, .5, 5.0)
        
        #pull actual test ratings
        test_actual_ratings = test_set['rating'].values
        
        rmse = np.sqrt(mean_squared_error(test_actual_ratings, v_mf_predictions))
        mae = mean_absolute_error(test_actual_ratings, v_mf_predictions)
        
        #calculate errors
        errors = test_actual_ratings, v_mf_predictions
        bias = np.mean(errors)
        variance = np.var(errors)
        
        #  % of predictions that were close
        within_half_star = np.mean(np.absolute(errors) <= .5) * 100
        within_one_star = np.mean(np.absolute(errors) <= 1.0) * 100
        
        #coverage
        coverage = np.mean(v_mf_predictions != vanilla_MF.global_mean) * 100
        
        metrics = {
                'rmse': rmse,
                'mae': mae,
                'bias': bias, 
                'variance': variance,
                'within_half_star': within_half_star,
                'within_one_star': within_one_star, 
                'coverage': coverage, 
        }
        
        return metrics
    
    def Precision_Recall_AtK(self, input_ratings_df, num_recommendations, k, relevance_threshold):

        # list to save precision@k and recall@k for each user
        precisions = []
        recalls = []

        # get all the userIds in the dataframe and save them in a list
        userId_list = list(input_ratings_df["userId"].unique())      

        # loop over all unique users
        for uid in userId_list:

            # get movie ratings dataframe for the user
            user_df = input_ratings_df[input_ratings_df["userId"] == uid]
            user_df.rename(columns={"rating":"y_rating"}, inplace=True)
            
            # filter to releavnt movies
            relevant_df = input_ratings_df[(input_ratings_df["userId"] == uid)&(input_ratings_df["rating"] >=relevance_threshold)]

            # skip users with 0 relevant ratings
            if relevant_df.shape[0] == 0:
                continue

            # clean relevant movies df
            relevant_df = relevant_df[["userId","movieId","rating"]]
            relevant_df.rename(columns={"rating":"y_rating"}, inplace=True)

            # get list of watched relevant movies
            watched_movies_list = relevant_df["movieId"].to_list()      # this may have to be passed to the function below

            user_recom_movies = self.recommend_movies(uid, n_recommendations = num_recommendations, exclude_already_watched=True)   #already_watched=watched_movies_list)
            pred_df = pd.DataFrame(user_recom_movies, columns=['movieId', 'rating'])
            pred_df.rename(columns={"rating":"pred_rating"}, inplace=True)

            # join user_df and pred_df
            pred_y_df = pd.merge(user_df, pred_df, on="movieId", how="right")

            # filter to items where there is actual rating
            pred_y_df_filtered = pred_y_df[pred_y_df["y_rating"].notna()]
            
            # sort dataframe descendingly by pred_rating
            pred_y_df_filtered.sort_values(by='pred_rating', ascending=False, inplace=True)

            #### compute precision@k ####
            # get df of recommended movies @ k
            recommended_movies_at_k = pred_y_df_filtered.iloc[:k]

            # get number of relevant items based on relevance threshold
            num_relevant_items = recommended_movies_at_k[recommended_movies_at_k["y_rating"] >= relevance_threshold].shape[0]
            if recommended_movies_at_k.shape[0] == 0:
                continue
            precision = num_relevant_items/recommended_movies_at_k.shape[0]

            # add precision to list 
            precisions.append(precision)

            #### compute recall@k ####
            # total number of relevant movies 
            total_relevant_items = pred_y_df_filtered[pred_y_df_filtered["y_rating"] >= relevance_threshold].shape[0]
            if total_relevant_items == 0:
                continue
            recall = num_relevant_items/total_relevant_items
            recalls.append(recall)

        if len(precisions) == 0:
            print("Length of Precision@ks is 0")
        else:
            self.precision_at_k = sum(precisions) / len(precisions)

        if len(recalls) == 0:
            print("Length of Recall@Ks is 0")
        else:
            self.recall_at_k = sum(recalls) / len(recalls)

        # code for sanity check
        return user_df, relevant_df, pred_y_df
    
    
   
        
            

### Pull Data and Split

In [57]:
#current_directory = Path.cwd()
# data_path = '../data/ml-32m/'

#Get current notebook directory
current_dir = Path(__file__).parent if "__file__" in globals() else Path.cwd()

# Define path relative to project root
data_path = current_dir.parent.parent / "data" /"ml-32m"


links_data_path = data_path / 'links.csv'
movies_data_path = data_path /'movies.csv'
ratings_data_path = data_path /  'ratings.csv'
tags_data_path = data_path / 'tags.csv'

#pull CSV files in

links_data = pd.read_csv(links_data_path) #infers whether it has columns or not
movies_data = pd.read_csv(movies_data_path) 
ratings_data = pd.read_csv(ratings_data_path) 
tags_data = pd.read_csv(tags_data_path) 

ratings_df = ratings_data.sample(n=150000, replace=False, random_state=158, axis = 0)
tags_df = tags_data.sample(n=150000, replace=False, random_state=158, axis = 0)

ratings_train_df, ratings_test_df = train_test_split(ratings_df, test_size = .2, random_state = 158)
tags_train_df, tags_test_df = train_test_split(tags_df, test_size = .2, random_state = 158)

### Fit Model

In [124]:

vanilla_MF = MF_Recommender(
                            n_factors = 25, 
                            learning_rate = .01, 
                            regularization = .01, 
                            epochs = 50, 
                            batch_size = 1024,
                            use_bias=True, 
                            use_latent_features = True,
                            use_content_features=False
            )

vanilla_MF.fit_model(ratings_train_df, tags_train_df)


Training started with Mini Batch SGD (batch size=1024)
Data has 66089 Total Users, 11964 Total movies, and 25 K dimensions
Total Epochs is 50, Learning rate is 0.01, and 94 batches per epoch

Starting fit for epoch 1/50
Finished fitting Epoch 1
Starting fit for epoch 11/50
Finished fitting Epoch 11
Starting fit for epoch 21/50
Finished fitting Epoch 21
Starting fit for epoch 31/50
Finished fitting Epoch 31
Starting fit for epoch 41/50
Finished fitting Epoch 41
Starting fit for epoch 50/50
Finished fitting Epoch 50
Finished all Epochs. Total time elapsed  0.32 minutes
Finished fitting. Total time elapsed  0.33 minutes
Epoch 50/50: Training RMSE loss is  0.764375 and Validation RMSE loss is  0.954075
Training MAE loss is  0.589370 and Validation MAE loss is  0.751300


### Evaluate Fit of Model

In [13]:
results = vanilla_MF.simple_evaluation(ratings_test_df)
print('\n')
print(f"Test Set Results: \n     RMSE: {results['rmse']:.4f} \n     MAE: {results['mae']: .4f}")
print(f"     Bias: {results['bias']:.4f} \n     Variance: {results['variance']: .4f}")
print(f"\n     Prediction Accuracy: \n     Within +- .5 stars {results['within_half_star']:.1f}% \n     Within +- 1 stars {results['within_one_star']:.1f}% \n")
print(f"     Coverage:\n     Valid Predictions: {results['coverage']:.1f}% ")




Test Set Results: 
     RMSE: 1.0022 
     MAE:  0.7826
     Bias: 3.5315 
     Variance:  0.6750

     Prediction Accuracy: 
     Within +- .5 stars 0.8% 
     Within +- 1 stars 2.2% 

     Coverage:
     Valid Predictions: 62.1% 


### Predict Movies

In [42]:
predict_user = 72743
top_k_movies = 10

recommended_movies = vanilla_MF.recommend_movies(predict_user, n_recommendations = top_k_movies)
print('\n')
print(f'Top {top_k_movies} Movie selections for UserId {predict_user}:')
print(f'\nRecommended Movies:')
print(f'MovieId     Rating')
for movie in range(top_k_movies):
    movieId, predicted_rating = recommended_movies[movie]
    print(f'{movieId:<10}: {predicted_rating: .2f}')



Top 10 Movie selections for UserId 72743:

Recommended Movies:
MovieId     Rating
127390    :  4.89
159817    :  4.59
86377     :  4.37
2068      :  4.33
5008      :  4.33
6993      :  4.33
26082     :  4.32
1232      :  4.31
198157    :  4.31
96606     :  4.30


# Sanity Checks

In [115]:
ratings_test_df.head()

,userId,movieId,rating,timestamp
30054596,188514,1285,5.0,961529127
5262037,32813,260,2.5,1400522119
13262391,82970,1079,3.0,1493316890
9495770,59358,46578,4.0,1580887858
26348111,165474,4896,4.5,1461096097


In [ ]:
# get frequency of movieId by userId
df = ratings_test_df
unique_movies_df = (
    df.groupby('userId')['movieId']
      .nunique()
      .reset_index(name='unique_movie_count')
      .sort_values(by='unique_movie_count', ascending=False)
)
unique_movies_df.head(15)

,userId,unique_movie_count
21021,175325,30
6717,55653,14
12548,103925,10
3244,26769,10
20599,171795,9
4525,37008,9
1617,13492,9
16220,133878,8
5979,49305,8
6368,52617,8


In [119]:
# slice test data to userId = 26769
ratings_test_df_26769 = ratings_test_df[ratings_test_df["userId"] == 26769]

In [125]:
# test precision@k and recall@k for userId = 26769
user_df_26769, relevant_df_26769, pred_y_df_26769 = vanilla_MF.Precision_Recall_AtK(ratings_test_df_26769, 10,10, 3)
vanilla_MF.precision_at_k, vanilla_MF.recall_at_k

Length of Precision@ks is 0
Length of Recall@Ks is 0


/var/folders/xj/r1x3kdf11q5bkkj160cjq1pw0000gn/T/ipykernel_36915/2734756994.py:552: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pred_y_df_filtered.sort_values(by='pred_rating', ascending=False, inplace=True)


(None, None)

In [ ]:
# prediction data joined with test data for userId = 26769
pred_y_df_26769

,userId,movieId,y_rating,timestamp,pred_rating
0,NaN,127390,NaN,NaN,4.649974
1,NaN,159817,NaN,NaN,4.351188
2,NaN,86377,NaN,NaN,4.129765
3,NaN,2068,NaN,NaN,4.093470
4,NaN,5008,NaN,NaN,4.091453
5,NaN,6993,NaN,NaN,4.088106
6,NaN,26082,NaN,NaN,4.085384
7,NaN,1232,NaN,NaN,4.071154
8,NaN,198157,NaN,NaN,4.068560
9,NaN,96606,NaN,NaN,4.060033


In [ ]:
# relevant test data for useId = 26769
relevant_df_26769

,userId,movieId,y_rating
4271802,26769,214722,4.0
4268200,26769,1784,3.5
4270880,26769,103980,3.0
4269911,26769,44199,4.0
4271079,26769,127152,3.0
4271527,26769,187231,4.0
4267906,26769,778,3.0
4270843,26769,101210,4.0


In [ ]:
# test data for userId = 26769
user_df_26769

,userId,movieId,y_rating,timestamp
4271802,26769,214722,4.0,1616269480
4268230,26769,1935,2.0,1371595337
4268200,26769,1784,3.5,1371393720
4270880,26769,103980,3.0,1394940654
4271752,26769,208205,2.0,1594483056
4269911,26769,44199,4.0,1371682331
4271079,26769,127152,3.0,1441504389
4271527,26769,187231,4.0,1625338894
4267906,26769,778,3.0,1371398897
4270843,26769,101210,4.0,1430323854
